# Test Depth Anything V3 with `DA3MONO-LARGE`

This notebook provides a complete environment setup and inference pipeline to test **Depth Anything V3** (DA3) using the official `depth-anything/DA3MONO-LARGE` model.

- **GitHub Repository**: [ByteDance-Seed/depth-anything-3](https://github.com/ByteDance-Seed/depth-anything-3)
- **Hugging Face Model**: [depth-anything/DA3MONO-LARGE](https://huggingface.co/depth-anything/DA3MONO-LARGE)

In [ ]:
# 1. Clone the repository and install depth-anything-3 package
import os

if not os.path.exists("depth-anything-3"):
    print("Cloning depth-anything-3 repository...")
    !git clone https://github.com/ByteDance-Seed/depth-anything-3.git
else:
    print("depth-anything-3 repository already exists.")

In [ ]:
# 2. Install the library in editable mode and additional dependencies
!pip install -e depth-anything-3
!pip install transformers accelerate diffusers peft pillow matplotlib opencv-python

## Load the Model and Run Monocular Depth Estimation

In [ ]:
import sys
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add depth-anything-3 directory to system path if needed
sys.path.append(os.path.abspath("depth-anything-3"))

from depth_anything_3.api import DepthAnything3

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load model from Hugging Face Hub (depth-anything/DA3MONO-LARGE)
model_id = "depth-anything/DA3MONO-LARGE"
print(f"Loading model: {model_id}...")
model = DepthAnything3.from_pretrained(model_id)
model = model.to(device=device)
print("Model loaded successfully!")

## Run Inference and Visualize Depth Estimation

In [ ]:
# Create a dummy image or specify a path to a local image in your test dataset
image_path = "sample_test.jpg"

if not os.path.exists(image_path):
    print("Creating a synthetic gradient image for test...")
    # Create a nice synthetic image with a gradient and simple shapes to check depth
    img = np.zeros((480, 640, 3), dtype=np.uint8)
    for y in range(480):
        img[y, :, 0] = int(y / 480 * 255)  # blue gradient
    cv2.circle(img, (320, 240), 100, (255, 255, 255), -1)  # white circle in center
    cv2.imwrite(image_path, img)
else:
    print(f"Using existing image: {image_path}")

In [ ]:
# Run inference
images = [image_path]
print(f"Predicting depth map for {image_path}...")
with torch.no_grad():
    prediction = model.inference(images)

# Display the output shape and details
print(f"Prediction object type: {type(prediction)}")
if hasattr(prediction, 'depth'):
    print(f"Depth tensor shape: {prediction.depth.shape}")
    depth_map = prediction.depth[0].cpu().numpy() if torch.is_tensor(prediction.depth) else prediction.depth[0]
else:
    # Fallback to key-based or index-based extraction if API varies
    print("Checking prediction attributes:", dir(prediction))
    depth_map = prediction[0]

# Visualize
orig_img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(orig_img)
plt.title("Input Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(depth_map, cmap="viridis")
plt.title("Depth Anything V3 Prediction (da3mono-large)")
plt.colorbar(label="Relative Depth")
plt.axis("off")

plt.tight_layout()
plt.show()